# Module 8: Synthetic Control

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Synthetic control builds a weighted average of untreated units that tracks the
treated unit before the intervention, and uses it as the counterfactual
afterwards. It is the most visually persuasive method in this series.

On this dataset it fails at all five treated agencies, **for a reason that is
visible before the answer is computed**, and the module is about learning to
look at that reason first.

**About 35 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

KEEP = [a for a in TRAINED if a != "A007"]     # the pre trend violator, Intermediate 8
BASELINE = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
d["base"] = d["agency_id"].map(BASELINE)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated, form=None, outcome="n_uof", offset=None):
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated))
                  & (s["period"] == "phase")).astype(float)
    fo = form or f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase"
    z = smf.glm(fo, s, family=sm.families.Poisson(),
                offset=s["lo"] if offset is None else offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

## 2. The method

> choose weights w over the untreated units, non negative and summing to one,
> that minimise the distance between the treated unit and the weighted donors
> **over the pre period**
>
> then read the gap between them after the intervention

The convexity constraint, weights non negative and summing to one, is what
keeps the synthetic unit interpretable and stops it extrapolating. It is also
the source of the failure here.

In [ ]:
from scipy.optimize import minimize

dd = f.copy()
piv = dd.pivot_table(index="year_month", columns="agency_id",
                     values="uof_per_100_arrests").interpolate()
pre = piv[piv.index < "2023-07"]
post = piv[piv.index >= "2023-11"]
donors = [a for a in piv.columns if a not in TRAINED]
print(f"  {len(donors)} donors, {len(pre)} pre period months")
print(f"  donor pre period means: "
      f"{', '.join(f'{NAME[a].split()[0]} {pre[a].mean():.2f}' for a in donors)}")

## 3. The diagnostic that comes first

Before fitting anything, compare each treated agency's pre period level
against the range the donors span. **A convex combination cannot go outside
the range of its inputs.**

In [ ]:
lo_d, hi_d = pre[donors].mean().min(), pre[donors].mean().max()
print(f"  the donors span {lo_d:.2f} to {hi_d:.2f} on the pre period mean\n")
for t in TRAINED:
    m = pre[t].mean()
    inside = lo_d <= m <= hi_d
    print(f"  {NAME[t]:34s} {m:.2f}   "
          f"{'inside the donor range' if inside else 'OUTSIDE the donor range'}")

**Four of the five sit above every donor.** No weighted average of the donors
can reach them, so the pre period fit is guaranteed to be poor before a single
weight is estimated. That is not a property of the optimiser; it is
arithmetic.

The fifth, Summit County at 3.12, does sit inside the donors' range. **Being
inside the range is necessary, not sufficient**, as the next cell shows: a
convex combination can match a level and still fail to track a path.

## 4. Fitting it anyway

In [ ]:
rows = []
for t in TRAINED:
    Y, X = pre[t].values, pre[donors].values
    r = minimize(lambda w: np.mean((Y - X @ w) ** 2),
                 np.repeat(1 / len(donors), len(donors)),
                 bounds=[(0, 1)] * len(donors),
                 constraints=({"type": "eq", "fun": lambda w: w.sum() - 1},))
    rmse = np.sqrt(np.mean((Y - X @ r.x) ** 2))
    eff = 100 * (post[t].mean() / (post[donors].values @ r.x).mean() - 1)
    top = ", ".join(f"{NAME[a].split()[0]} {x:.2f}"
                    for a, x in sorted(zip(donors, r.x), key=lambda z: -z[1])[:2])
    rows.append({"agency": NAME[t].split()[0],
                 "pre period fit error": f"{100 * rmse / Y.mean():.0f}%",
                 "what it reports": f"{eff:+.1f}%",
                 "largest weights": top})
print(f"  the true effect at every one of these is {TRUTH:+.1f}%\n")
pd.DataFrame(rows).set_index("agency")

Five agencies, one true effect of 12 percent, answers spanning from **plus 11
to minus 25 percent.**

And every one of them carries a pre period fit error between 25 and 48 percent
of the agency's own mean, **including Summit County, whose level was inside
the donors' range.** Its 29 percent error is the second worst of the five: the
donors can reach its average and cannot follow its steeper decline. **A synthetic control whose pre period fit error is
a quarter of the outcome's level has not been fitted; it has been declared.**

The one that comes closest to the truth is Summit County, and it gets there
for the wrong reason: the pre trend violation from
[Intermediate Module 8](../../Intermediate/Notebooks/Module_08_When_Parallel_Trends_Fails.ipynb)
is being read as an effect.

## 5. The diagnostics, in the order to run them

| Check | Threshold | What it catches |
|---|---|---|
| **Is the treated unit inside the donors' range** | it must be | the failure here |
| Pre period fit error relative to the outcome's level | under about 5 percent | a synthetic unit that does not track |
| Number of donors receiving weight | more than two or three | a fit driven by one lucky donor |
| Placebo on each donor in turn | the treated gap should be extreme | a gap that is ordinary |
| Pre period length | long enough to identify the weights | overfitting the pre period |

**The first row is free and it settles this case.** Reporting an effect from a
synthetic control without the pre period fit error is like reporting a
regression coefficient without its standard error.

## 6. When synthetic control is the right tool

| Situation | Verdict |
|---|---|
| One treated unit, many donors, long pre period | its home ground |
| The treated unit sits inside the donor distribution | required |
| Donors are plausibly unaffected by the treatment | required |
| Several treated units | possible, but difference in differences is usually simpler |
| The treated unit is the largest or most extreme | **do not**; this module is the reason |

The last row is common in public safety work, because the agency that gets a
program is often the one with the most extreme numbers. **Selection on the
outcome and synthetic control are a bad combination**, and the first is
routine.

## Exercise

Allow the weights to be negative, dropping the convexity constraint, and see
whether the fit problem goes away and what it costs.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    rows = []
    for t in TRAINED:
        Y, X = pre[t].values, pre[donors].values
        conv = minimize(lambda w: np.mean((Y - X @ w) ** 2),
                        np.repeat(1 / len(donors), len(donors)),
                        bounds=[(0, 1)] * len(donors),
                        constraints=({"type": "eq",
                                      "fun": lambda w: w.sum() - 1},))
        free = np.linalg.lstsq(X, Y, rcond=None)[0]
        rows.append({
            "agency": NAME[t].split()[0],
            "fit error, convex": f"{100 * np.sqrt(np.mean((Y - X @ conv.x) ** 2)) / Y.mean():.0f}%",
            "fit error, unconstrained": f"{100 * np.sqrt(np.mean((Y - X @ free) ** 2)) / Y.mean():.0f}%",
            "convex reports": f"{100 * (post[t].mean() / (post[donors].values @ conv.x).mean() - 1):+.1f}%",
            "unconstrained reports": f"{100 * (post[t].mean() / (post[donors].values @ free).mean() - 1):+.1f}%",
            "largest absolute weight": round(np.abs(free).max(), 2)})
    print(f"  the truth is {TRUTH:+.1f}% everywhere\n")
    display(pd.DataFrame(rows).set_index("agency"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

Dropping the constraint improves the pre period fit at every agency, and the
weights become large and of both signs.

**The fit improved because the method is now extrapolating**, which is exactly
what the convexity constraint exists to forbid. A synthetic unit built from
weights of plus two and minus one is not a weighted average of real agencies;
it is a linear extrapolation dressed as one, and its behaviour after the
intervention rests on the extrapolation continuing to hold.

Abadie's framing is worth stating plainly: the constraint is a **safeguard
against extrapolation**, and a treated unit outside the donors' range is the
method telling you that no safe counterfactual exists in this donor pool. The
answer is a better donor pool, not a looser constraint.

For this dataset the better donor pool would have to contain agencies with use
of force rates above 3.5 per 100 arrests, and the five that had them all took
the program.

</details>

---

**Next:** [Module 9: Honest Inference with Few Clusters](Module_09_Honest_Inference_With_Few_Clusters.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*